# 01 — Data Ingestion

Load UCI Online Retail II (two sheets), concat, inspect schema.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_settings
from src.data import load_raw

In [2]:
settings = load_settings(Path.cwd().parent / "config" / "settings.yaml")
xlsx_path = Path.cwd().parent / settings["paths"]["bronze_source"]
xlsx_path

WindowsPath('d:/disertatie v2/Customer Behavior Analysis/data/bronze/online_retail_II.xlsx')

In [3]:
df = load_raw(xlsx_path)
df.shape

(1067371, 8)

## Schema

In [4]:
df.dtypes

Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

In [5]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Nulls

In [6]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

## Date range

In [7]:
df["InvoiceDate"].min(), df["InvoiceDate"].max()

(Timestamp('2009-12-01 07:45:00'), Timestamp('2011-12-09 12:50:00'))

## Clean transactions

In [8]:
from src.data import clean_transactions

df_clean = clean_transactions(
    df,
    drop_missing_customer=True,
    excluded_stockcodes=tuple(settings["excluded_stockcodes"]),
    country_filter=settings["country_filter"],
)
df_clean.shape

start: 1067371 rows
after strip+upper: 1067371 rows
after drop null Customer ID: 824364 rows
after Price > 0: 824293 rows
after exclude stockcodes: 820890 rows
after country=United Kingdom: 739914 rows
after dedupe: 714764 rows


(714764, 8)

## Add revenue

In [9]:
from src.data import add_revenue, split_purchases_returns

df_clean = add_revenue(df_clean)
df_clean.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


## Split purchases / returns

In [10]:
purchases, returns = split_purchases_returns(df_clean)
len(purchases), len(returns)

(699681, 15083)

## Save parquets

In [11]:
bronze_dir = Path.cwd().parent / settings["paths"]["bronze"]
silver_dir = Path.cwd().parent / settings["paths"]["silver"]

df.to_parquet(bronze_dir / "transactions_concat.parquet", index=False)
df_clean.to_parquet(silver_dir / "transactions_clean.parquet", index=False)

print(f"bronze: {bronze_dir / 'transactions_concat.parquet'}")
print(f"silver: {silver_dir / 'transactions_clean.parquet'}")

bronze: d:\disertatie v2\Customer Behavior Analysis\data\bronze\transactions_concat.parquet
silver: d:\disertatie v2\Customer Behavior Analysis\data\silver\transactions_clean.parquet
